In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.stats.stattools import durbin_watson

cs = pd.read_excel('/Users/elyas/vscode/market_analysis_03_office/data/DataExport_office_v2.xlsx')
ff = pd.read_excel('../data/FEDFUNDS (1).xlsx')
bond = pd.read_excel('/Users/elyas/vscode/market_analysis_03_office/data/bond_yield_10yr.xlsx')
rrp = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/real_risk_premieum.csv')
inf_e = pd.read_excel('/Users/elyas/vscode/market_analysis_03_office/data/Inflation expectations (1).xlsx')
cmbs = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/commercial_mortgage.csv')
gdp = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/GDP.csv')

#create a subset where column Population is greater than 500000
cs = cs[cs['Population'] > 500000]

#convert to time series
def convert_period_to_datetime(period):
	parts = period.split('Q')
	year = parts[0].strip()
	quarter = parts[1].strip().split()[0]
	month = (int(quarter) - 1) * 3 + 1
	return pd.Timestamp(year=int(year), month=month, day=1)

cs['Period'] = cs['Period'].apply(convert_period_to_datetime)

#remove periods in the year 2025
cs = cs[cs['Period'].dt.year != 2025]

#get data only where geography has Atlanta
cs = cs[cs['Geography Name'].str.contains('Atlanta')]

#change the column name to period
cmbs.rename(columns={'observation_date': 'period'}, inplace=True)
gdp.rename(columns={'observation_date': 'period'}, inplace=True)
cmbs.rename(columns={'ASCMA': 'oustanding_mortgage'}, inplace=True)
gdp.rename(columns={'GDP': 'gdp'}, inplace=True)
rrp.rename(columns={'observation_date': 'period'}, inplace=True)
inf_e.rename(columns={'Model Output Date': 'period'}, inplace=True)


#the units in outstanding mortgage is in millions, let's convert it to billions
cmbs['oustanding_mortgage'] = cmbs['oustanding_mortgage'] / 1000

#create a new column in cmbs called cmbs_to_gdp
cmbs['cmbs_to_gdp'] = cmbs['oustanding_mortgage'] / gdp['gdp']


#convert fed funds rate to quarterly
ff['period'] = pd.to_datetime(ff['period'])
ff.set_index('period', inplace=True)
ff = ff.resample('Q').mean()
#each data should start on first of the quarter
ff.index = ff.index.to_period('Q').start_time
#change the column period from index to column
ff.reset_index(inplace=True)


#convert real risk premieum to quarterly
inf_e['period'] = pd.to_datetime(inf_e['period'])
inf_e.set_index('period', inplace=True)
inf_e = inf_e.resample('Q').mean()
#change the variable TENEXPCHAREARISPRE to ten_yr_risk_premium
inf_e.rename(columns={'TENEXPCHAREARISPRE': 'ten_yr_risk_premium'}, inplace=True)
#each data should start on first of the quarter
inf_e.index = inf_e.index.to_period('Q').start_time
#change the column period from index to column
inf_e.reset_index(inplace=True)

#convert real risk premieum to quarterly
rrp['period'] = pd.to_datetime(rrp['period'])
rrp.set_index('period', inplace=True)
rrp = rrp.resample('Q').mean()
#change the variable TENEXPCHAREARISPRE to ten_yr_risk_premium
rrp.rename(columns={'TENEXPCHAREARISPRE': 'ten_yr_risk_premium'}, inplace=True)
#each data should start on first of the quarter
rrp.index = rrp.index.to_period('Q').start_time
#change the column period from index to column
rrp.reset_index(inplace=True)

#get data from 2000
inf_e = inf_e[inf_e['period'] >= '2000-01-01']

#Get the data we want

cs = cs[
    [
        "Geography Name",
        "Period",
        "Appreciation Return",
        "Availability Rate",
        "Available SF Direct",
        "Average Sale Price",
        "Cap Rate",
        "Market Cap Rate",
        "Construction Starts SF",
        "Construction Starts SF 12 Mo",
        "Demand SF",
        "Demolished SF",
        "Gross Delivered SF",
        "Inventory SF",
        "Leasing SF Total",
        "Market Asking Rent Growth",
        "Market Asking Rent Growth 12 Mo",
        "Net Absorption SF",
        "Net Absorption SF 12 Mo",
        "Net Delivered SF",
        "Net Delivered SF 12 Mo",
        "Occupancy Rate",
        "Sales Volume Transactions",
        "Sold Building SF",
        "Vacancy Rate",
        "Under Construction SF",
        "Total Sales Transactions"
    ]
].copy()

# Next, rename columns
cs.rename(
    columns={
        "Geography Name": "geography",
        "Period": "period",
        "Appreciation Return": "appreciation_return",
        "Availability Rate": "availability_rate",
        "Available SF Direct": "available_df_direct",
        "Average Sale Price": "avg_sale_price",
        "Cap Rate": "cap_rate",
        "Market Cap Rate": "market_cap_rate",
        "Construction Starts SF": "starts_sf",
        "Construction Starts SF 12 Mo": "starts_sf_12_mo",
        "Demand SF": "demand_sf",
        "Demolished SF": "demolished_sf",
        "Gross Delivered SF": "gross_delivered_sf",
        "Inventory SF": "inventory_sf",
        "Leasing SF Total": "leasing_sf_total",
        "Market Asking Rent Growth": "asking_rent_growth",
        "Market Asking Rent Growth 12 Mo": "asking_rent_growth_12_mo",
        "Net Absorption SF": "net_absorp_sf",
        "Net Absorption SF 12 Mo": "net_absorp_sf_12_mo",
        "Net Delivered SF": "net_delivered_sf",
        "Net Delivered SF 12 Mo": "net_delivered_sf_12_mo",
        "Occupancy Rate": "occupancy_rate",
        "Sales Volume Transactions": "sales_volume",
        "Sold Building SF": "sold_building_sf",
        "Vacancy Rate": "vacancy_rate",
        "Under Construction SF": "under_construction_sf",
        "Total Sales Transactions": "total_sales_transactions"
    },
    inplace=True
)


### merge datasets

cmbs['period'] = pd.to_datetime(cmbs['period'])
cs = pd.merge(cs, bond, on='period', how='left')
cs = pd.merge(cs, inf_e, on='period', how='left')
cs = pd.merge(cs, rrp, on='period', how='left')
cs = pd.merge(cs, ff, on='period', how='left')
cs = pd.merge(cs, cmbs[['period', 'cmbs_to_gdp']], on='period', how='left')
cs = pd.merge(cs, bond, on='period', how='left')


#convert variables to percentage
#multiply by 100 availability rate
cs['availability'] = cs['availability_rate'] * 100
cs['cap_rate'] = cs['cap_rate'] * 100
cs['market_cap_rate'] = cs['market_cap_rate'] * 100
cs['asking_rent_growth'] = cs['asking_rent_growth'] * 100
cs['asking_rent_growth_12_mo'] = cs['asking_rent_growth_12_mo'] * 100
cs['occupancy_rate'] = cs['occupancy_rate'] * 100
cs['vacancy_rate'] = cs['vacancy_rate'] * 100
cs['appreciation_return'] = cs['appreciation_return'] * 100
cs['one_yr_inf_exp'] = cs['one_yr_inf_exp'] * 100
cs['two_yr_inf_exp'] = cs['two_yr_inf_exp'] * 100
cs['three_yr_inf_exp'] = cs['three_yr_inf_exp'] * 100
cs['four_yr_inf_exp'] = cs['four_yr_inf_exp'] * 100
cs['five_yr_inf_exp'] = cs['five_yr_inf_exp'] * 100
cs['six_yr_inf_exp'] = cs['six_yr_inf_exp'] * 100
cs['seven_yr_inf_exp'] = cs['seven_yr_inf_exp'] * 100
cs['eight_yr_inf_exp'] = cs['eight_yr_inf_exp'] * 100
cs['nine_yr_inf_exp'] = cs['nine_yr_inf_exp'] * 100
cs['ten_yr_inf_exp'] = cs['ten_yr_inf_exp'] * 100
cs['cmbs_to_gdp'] = cs['cmbs_to_gdp'] * 100

df = cs.copy()


/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_4414/284052440.py:55: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  ff = ff.resample('Q').mean()
/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_4414/284052440.py:65: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  inf_e = inf_e.resample('Q').mean()
/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_4414/284052440.py:76: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  rrp = rrp.resample('Q').mean()


In [2]:
#export data to csv
df.to_csv('../data/office_data_atlanta_time-series.csv', index=False)

In [3]:
#multiple sales_volume by 1000
#create a variable where sales_volume is divided by total_sales_transactions
cs['sales_volume_per_transaction'] = cs['sales_volume'] / cs['total_sales_transactions']

In [4]:
inf_e.head()

,period,one_yr_inf_exp,two_yr_inf_exp,three_yr_inf_exp,four_yr_inf_exp,five_yr_inf_exp,six_yr_inf_exp,seven_yr_inf_exp,eight_yr_inf_exp,nine_yr_inf_exp,ten_yr_inf_exp
72,2000-01-01,0.031587,0.032726,0.033114,0.033276,0.033338,0.033347,0.033325,0.033285,0.033232,0.033173
73,2000-04-01,0.034112,0.033621,0.033384,0.033220,0.033086,0.032970,0.032864,0.032767,0.032676,0.032590
74,2000-07-01,0.031280,0.031358,0.031175,0.031001,0.030860,0.030749,0.030661,0.030588,0.030528,0.030477
75,2000-10-01,0.031242,0.030688,0.030253,0.029950,0.029741,0.029593,0.029487,0.029410,0.029353,0.029311
76,2001-01-01,0.024760,0.025448,0.025456,0.025406,0.025380,0.025385,0.025413,0.025458,0.025515,0.025579


In [5]:
df.shape

(100, 43)

In [6]:
## Step 0 Removing the null values
#forwardfill cmbs_to_gdp
df['cmbs_to_gdp'] = df['cmbs_to_gdp'].fillna(method='ffill')

#impute the missine value for the cap rate using the mean of the value of cap rate where period is 2009-01-01 and 2009-07-01
cap_rate_mean = df[(df['period'] == '2009-01-01') | (df['period'] == '2009-07-01')]['cap_rate'].mean()
df['cap_rate'].fillna(cap_rate_mean, inplace=True)

#drop the columns availability, availability_rate, and available_df_direct
df.drop(columns=['availability', 'availability_rate', 'available_df_direct'], inplace=True)

/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_4414/1447694489.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['cmbs_to_gdp'] = df['cmbs_to_gdp'].fillna(method='ffill')
/var/folders/5t/k32b1zys181cv4n2nz_7rq1c0000gp/T/ipykernel_4414/1447694489.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['cap_rate'].fillna(cap_rate_mean, inplace=True)


In [7]:
## Step 0.1 Check for multicollinearity
#check for multicollinearity
### Check for multicollinearity

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

def calculate_vif(df):
    """Calculate VIF for each variable in the dataset."""
    vif_data = pd.DataFrame()
    vif_data["Variable"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data

# Calculate VIF for the filtered dataset (excluding the target variable)
X = df.drop(columns=['appreciation_return'])  # predictors only
# select only numeric columns to avoid type errors in VIF calculation
X_numeric = X.select_dtypes(include=[np.number])
vif_df = calculate_vif(X_numeric)
print(vif_df)


                    Variable           VIF
0             avg_sale_price  1.080274e+01
1                   cap_rate  4.241658e+00
2            market_cap_rate  2.685505e+01
3                  starts_sf  4.101369e+00
4            starts_sf_12_mo  4.828758e+01
5                  demand_sf  7.689173e+03
6              demolished_sf           inf
7         gross_delivered_sf           inf
8               inventory_sf  8.392853e+03
9           leasing_sf_total  7.513537e+00
10        asking_rent_growth  6.966060e+00
11  asking_rent_growth_12_mo  2.094110e+01
12             net_absorp_sf  3.391043e+00
13       net_absorp_sf_12_mo  8.021481e+00
14          net_delivered_sf           inf
15    net_delivered_sf_12_mo  1.692670e+01
16            occupancy_rate  2.579622e+05
17              sales_volume  3.401884e+01
18          sold_building_sf  1.485219e+01
19              vacancy_rate  2.755254e+04
20     under_construction_sf  4.515913e+01
21  total_sales_transactions  4.310930e+01
22         

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [8]:
#print the columns with VIF is less than 5
print(vif_df[vif_df['VIF'] < 5])

         Variable       VIF
1        cap_rate  4.241658
3       starts_sf  4.101369
12  net_absorp_sf  3.391043


In [9]:
# Step 1. Data Preparation
# -----------------------------
# (a) Set the time index and filter the DataFrame to include only the selected variables.
# 'period' is already set as the index and assumed to be in datetime format.
# = df.set_index('period')
# Using all variables; no need to filter columns.

# (b) Define your target variable.
target_variable = 'market_cap_rate'  # This is the variable we want to predict.

# Step 2. Data Transformation & Stationarity Checks
# -----------------------------
# We will perform the following transformations:
#   A. Variance Stabilization via log-transformation (when appropriate)
#   B. First Differencing to help achieve stationarity
#   C. Seasonal Differencing (if needed)
#   D. Final selection of stationary series

# A. Variance Stabilization
cv_threshold = 0.5  # threshold for applying log transform
# We'll keep track of how we transform the target variable.
target_transformation = None

# Create a new DataFrame to hold the transformed columns.
df_transformed = pd.DataFrame(index=df.index)

# Loop through each numeric column
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols:
    series = df[col].dropna()
    if series.min() > 0:
        cv = series.std() / series.mean()
        if cv > cv_threshold:
            # Apply log transformation if coefficient of variation is high
            df_transformed[f'log_{col}'] = np.log(series)
            if col == target_variable:
                target_transformation = f'log_{col}'
        else:
            # Otherwise, keep the original series (rename for clarity)
            df_transformed[f'orig_{col}'] = series
            if col == target_variable:
                target_transformation = f'orig_{col}'
    else:
        # If the series contains non-positive values, keep it unchanged
        df_transformed[f'orig_{col}'] = series
        if col == target_variable:
            target_transformation = f'orig_{col}'

# B. First Differencing to remove trends (if series is non-stationary)
df_diff = df_transformed.copy()  # start with the transformed data

# For each transformed column, check stationarity using the ADF test and difference if necessary.
transformed_cols = [col for col in df_transformed.columns if col.startswith(('log_', 'orig_'))]
for col in transformed_cols:
    series = df_transformed[col].dropna()
    adf_pvalue = adfuller(series)[1]
    if adf_pvalue > 0.05:
        # The series appears non-stationary, so take first difference.
        base_name = col.split('_', 1)[1]
        df_diff[f'd1_{base_name}'] = df_transformed[col].diff()
        # For the target variable, update transformation name if differencing is applied.
        if base_name == target_variable:
            target_transformation = f'd1_{base_name}'
    else:
        # Series is stationary; copy it as is.
        df_diff[col] = df_transformed[col]

# Remove NA values generated by differencing.
df_diff = df_diff.dropna()

# C. Seasonal Differencing (if necessary)
seasonal_period = 8  # adjust this to your seasonality (e.g., quarterly data might use 4; here 8 is just an example)
df_seasonal = df_diff.copy()
differenced_cols = [col for col in df_diff.columns if col.startswith('d1_')]

for col in differenced_cols:
    series = df_diff[col].dropna()
    try:
        adf_pvalue = adfuller(series)[1]
        if adf_pvalue > 0.05:
            # Apply seasonal differencing if the series still appears non-stationary.
            df_seasonal[f'sd{seasonal_period}_{col}'] = series.diff(seasonal_period)
    except Exception as e:
        print(f"Skipping seasonal differencing for {col} due to error: {e}")

# Remove any additional NAs.
df_seasonal = df_seasonal.dropna()

In [10]:
#check missing values and print all
missing_values = df_seasonal.isnull().sum()
missing_values[missing_values > 0]


Series([], dtype: int64)

In [11]:
# D. Final Stationary Selection
# We select only the series that are now stationary based on the ADF test.
final_columns = []
for col in df_seasonal.columns:
    # We expect our stationary series to have names starting with either 'd1_' or 'sd{seasonal_period}_'
    if col.startswith(f'sd{seasonal_period}_') or col.startswith('d1_'):
        try:
            p_value = adfuller(df_seasonal[col].dropna())[1]
            if p_value < 0.05:
                final_columns.append(col)
        except Exception as e:
            print(f"ADF test could not be performed for {col}: {e}")

# Ensure that our target variable (after transformation) is included.
if target_transformation not in final_columns:
    final_columns.append(target_transformation)

# Final stationary DataFrame ready for further analysis.
stationary_df = df_seasonal[final_columns].dropna()

In [12]:
# Step 4 Granger Causality Testing

candidate_vars = [col for col in stationary_df.columns if col != target_transformation]
significant_vars = []
results_dict = {}  # to store p-values for each variable

maxlag = 8       # maximum lag to test
p_threshold = 0.05  # significance level

for var in candidate_vars:
    print(f"\nTesting Granger causality for variable: {var}")
    try:
        # Test if candidate variable Granger-causes the target variable.
        test_result = grangercausalitytests(
            stationary_df[[target_transformation, var]], 
            maxlag=maxlag, 
            verbose=False
        )
        p_values = []
        # Extract and print the p-value from the F-test for each lag.
        for i in range(maxlag):
            p_val = test_result[i+1][0]['ssr_ftest'][1]
            p_values.append(p_val)
            print(f"  Lag {i+1}: p-value = {p_val:.4f}")
        
        results_dict[var] = p_values
        # Determine if the variable is significant based on the minimum p-value.
        if min(p_values) < p_threshold:
            print(f"  => {var} is SIGNIFICANT (min p-value = {min(p_values):.4f})")
            significant_vars.append(var)
        else:
            print(f"  => {var} is NOT significant (min p-value = {min(p_values):.4f})")
    except Exception as e:
        print(f"  Error testing Granger causality for {var}: {e}")

# Summarize the p-values for each candidate variable.
print("\nSummary of Granger causality test results:")
for var, p_vals in results_dict.items():
    print(f"  {var}: {p_vals}")


Testing Granger causality for variable: d1_appreciation_return
  Lag 1: p-value = 0.0000
  Lag 2: p-value = 0.0000
  Lag 3: p-value = 0.0000
  Lag 4: p-value = 0.0000
  Lag 5: p-value = 0.0000
  Lag 6: p-value = 0.0000
  Lag 7: p-value = 0.0000
  Lag 8: p-value = 0.0000
  => d1_appreciation_return is SIGNIFICANT (min p-value = 0.0000)

Testing Granger causality for variable: d1_cap_rate
  Lag 1: p-value = 0.7126
  Lag 2: p-value = 0.7661
  Lag 3: p-value = 0.9640
  Lag 4: p-value = 0.9290
  Lag 5: p-value = 0.9089
  Lag 6: p-value = 0.9591
  Lag 7: p-value = 0.9402
  Lag 8: p-value = 0.8125
  => d1_cap_rate is NOT significant (min p-value = 0.7126)

Testing Granger causality for variable: d1_starts_sf
  Lag 1: p-value = 0.9970
  Lag 2: p-value = 0.1141
  Lag 3: p-value = 0.0400
  Lag 4: p-value = 0.0304
  Lag 5: p-value = 0.1080
  Lag 6: p-value = 0.1003
  Lag 7: p-value = 0.3182
  Lag 8: p-value = 0.4232
  => d1_starts_sf is SIGNIFICANT (min p-value = 0.0304)

Testing Granger causali

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  Lag 1: p-value = 0.2519
  Lag 2: p-value = 0.1792
  Lag 3: p-value = 0.2765
  Lag 4: p-value = 0.2303
  Lag 5: p-value = 0.5007
  Lag 6: p-value = 0.2711
  Lag 7: p-value = 0.3843
  Lag 8: p-value = 0.2738
  => d1_starts_sf_12_mo is NOT significant (min p-value = 0.1792)

Testing Granger causality for variable: d1_demand_sf
  Lag 1: p-value = 0.0269
  Lag 2: p-value = 0.5673
  Lag 3: p-value = 0.5694
  Lag 4: p-value = 0.6509
  Lag 5: p-value = 0.6337
  Lag 6: p-value = 0.8007
  Lag 7: p-value = 0.9576
  Lag 8: p-value = 0.9679
  => d1_demand_sf is SIGNIFICANT (min p-value = 0.0269)

Testing Granger causality for variable: d1_gross_delivered_sf
  Lag 1: p-value = 0.7123
  Lag 2: p-value = 0.3111
  Lag 3: p-value = 0.3975
  Lag 4: p-value = 0.4251
  Lag 5: p-value = 0.4279
  Lag 6: p-value = 0.3456
  Lag 7: p-value = 0.6251
  Lag 8: p-value = 0.6333
  => d1_gross_delivered_sf is NOT significant (min p-value = 0.3111)

Testing Granger causality for variable: d1_inventory_sf
  Lag 1: p-

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications

  Lag 1: p-value = 0.8606
  Lag 2: p-value = 0.3499
  Lag 3: p-value = 0.0670
  Lag 4: p-value = 0.1448
  Lag 5: p-value = 0.2217
  Lag 6: p-value = 0.2790
  Lag 7: p-value = 0.5826
  Lag 8: p-value = 0.5132
  => d1_net_delivered_sf is NOT significant (min p-value = 0.0670)

Testing Granger causality for variable: d1_net_delivered_sf_12_mo
  Lag 1: p-value = 0.0589
  Lag 2: p-value = 0.3289
  Lag 3: p-value = 0.7417
  Lag 4: p-value = 0.3239
  Lag 5: p-value = 0.2439
  Lag 6: p-value = 0.2534
  Lag 7: p-value = 0.6696
  Lag 8: p-value = 0.8059
  => d1_net_delivered_sf_12_mo is NOT significant (min p-value = 0.0589)

Testing Granger causality for variable: d1_occupancy_rate
  Lag 1: p-value = 0.0411
  Lag 2: p-value = 0.8537
  Lag 3: p-value = 0.3261
  Lag 4: p-value = 0.2491
  Lag 5: p-value = 0.1946
  Lag 6: p-value = 0.1296
  Lag 7: p-value = 0.4040
  Lag 8: p-value = 0.6944
  => d1_occupancy_rate is SIGNIFICANT (min p-value = 0.0411)

Testing Granger causality for variable: d1_sales

...

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  Lag 1: p-value = 0.3378
  Lag 2: p-value = 0.9274
  Lag 3: p-value = 0.7543
  Lag 4: p-value = 0.6755
  Lag 5: p-value = 0.5072
  Lag 6: p-value = 0.6665
  Lag 7: p-value = 0.8760
  Lag 8: p-value = 0.8536
  => d1_five_yr_inf_exp is NOT significant (min p-value = 0.3378)

Testing Granger causality for variable: d1_six_yr_inf_exp
  Lag 1: p-value = 0.3605
  Lag 2: p-value = 0.9417
  Lag 3: p-value = 0.7659
  Lag 4: p-value = 0.7160
  Lag 5: p-value = 0.5243
  Lag 6: p-value = 0.6769
  Lag 7: p-value = 0.8797
  Lag 8: p-value = 0.8676
  => d1_six_yr_inf_exp is NOT significant (min p-value = 0.3605)

Testing Granger causality for variable: d1_seven_yr_inf_exp
  Lag 1: p-value = 0.3793
  Lag 2: p-value = 0.9504
  Lag 3: p-value = 0.7759
  Lag 4: p-value = 0.7473
  Lag 5: p-value = 0.5366
  Lag 6: p-value = 0.6832
  Lag 7: p-value = 0.8817
  Lag 8: p-value = 0.8759
  => d1_seven_yr_inf_exp is NOT significant (min p-value = 0.3793)

Testing Granger causality for variable: d1_eight_yr_inf_e

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  Lag 1: p-value = 0.4185
  Lag 2: p-value = 0.9610
  Lag 3: p-value = 0.7968
  Lag 4: p-value = 0.8056
  Lag 5: p-value = 0.5591
  Lag 6: p-value = 0.6924
  Lag 7: p-value = 0.8843
  Lag 8: p-value = 0.8873
  => d1_ten_yr_inf_exp is NOT significant (min p-value = 0.4185)

Testing Granger causality for variable: d1_fed_funds_rate
  Lag 1: p-value = 0.8613
  Lag 2: p-value = 0.1631
  Lag 3: p-value = 0.2044
  Lag 4: p-value = 0.0253
  Lag 5: p-value = 0.0461
  Lag 6: p-value = 0.0156
  Lag 7: p-value = 0.0571
  Lag 8: p-value = 0.0323
  => d1_fed_funds_rate is SIGNIFICANT (min p-value = 0.0156)

Testing Granger causality for variable: d1_cmbs_to_gdp
  Lag 1: p-value = 0.0862
  Lag 2: p-value = 0.5875
  Lag 3: p-value = 0.4380
  Lag 4: p-value = 0.3875
  Lag 5: p-value = 0.3590
  Lag 6: p-value = 0.0803
  Lag 7: p-value = 0.0393
  Lag 8: p-value = 0.0836
  => d1_cmbs_to_gdp is SIGNIFICANT (min p-value = 0.0393)

Testing Granger causality for variable: d1_bond_yield_10yr_y
  Lag 1: p-valu

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


In [13]:
#print variabels that are significant
print("\nSignificant variables based on Granger causality tests:")
for var in significant_vars:
    print(f"  {var}")



Significant variables based on Granger causality tests:
  d1_appreciation_return
  d1_starts_sf
  d1_demand_sf
  d1_inventory_sf
  d1_asking_rent_growth
  d1_asking_rent_growth_12_mo
  d1_net_absorp_sf_12_mo
  d1_occupancy_rate
  d1_vacancy_rate
  d1_fed_funds_rate
  d1_cmbs_to_gdp


In [14]:
# Build the filtered DataFrame including the target variable and the selected predictors.
selected_vars = [
    'd1_appreciation_return',
    'd1_starts_sf',
    'd1_asking_rent_growth',
    'd1_bond_yield_10yr_x',
    'd1_fed_funds_rate',
    'd1_cmbs_to_gdp'
]

# Use target_transformation which corresponds to market_cap_rate.
filtered_df = stationary_df[[target_transformation] + selected_vars]

print(f"\nFiltered dataset shape: {filtered_df.shape}")
print(f"Variables included: {filtered_df.columns.tolist()}")


Filtered dataset shape: (91, 7)
Variables included: ['orig_market_cap_rate', 'd1_appreciation_return', 'd1_starts_sf', 'd1_asking_rent_growth', 'd1_bond_yield_10yr_x', 'd1_fed_funds_rate', 'd1_cmbs_to_gdp']


In [15]:
# Build the filtered DataFrame including the target and only the significant predictors.
#available_significant_vars = [var for var in significant_vars if var in stationary_df.columns]
#filtered_df = stationary_df[[target_transformation] + available_significant_vars]

#print(f"\nFiltered dataset shape: {filtered_df.shape}")
#print(f"Variables included: {filtered_df.columns.tolist()}")

In [16]:
### Check for multicollinearity

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

def calculate_vif(df):
    """Calculate VIF for each variable in the dataset."""
    vif_data = pd.DataFrame()
    vif_data["Variable"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data

# Calculate VIF for the filtered dataset (excluding the target variable)
X = filtered_df.drop(columns=[target_transformation])  # predictors only
vif_df = calculate_vif(X)
print(vif_df)


                 Variable       VIF
0  d1_appreciation_return  1.197215
1            d1_starts_sf  1.025740
2   d1_asking_rent_growth  1.071609
3    d1_bond_yield_10yr_x  1.217369
4       d1_fed_funds_rate  1.478440
5          d1_cmbs_to_gdp  1.491052


In [17]:
filtered_df.head()

,orig_market_cap_rate,d1_appreciation_return,d1_starts_sf,d1_asking_rent_growth,d1_bond_yield_10yr_x,d1_fed_funds_rate,d1_cmbs_to_gdp
9,9.740870,1.602388,-0.203654,0.551561,0.023333,0.009569,0.145904
10,9.588397,1.071423,-1.167049,-0.162926,-0.840000,-0.005731,0.149022
11,9.559014,1.226888,0.638993,-0.961229,-0.253333,-0.186930,0.240701
12,9.454589,-0.001805,0.294986,3.303083,-0.086667,-0.143812,0.082253
13,9.368833,0.553867,0.184572,-4.681604,-0.300000,-0.002670,0.148263


In [18]:
#get filtered_df for the following variables only d1_appreciation_return.L1 d1_appreciation_return.L3d1_appreciation_return.L4 d1_starts_sf_12_mo.L2 d1_demand_sf.L0 d1_demand_sf.L3 d1_demand_sf.L4 d1_net_absorp_sf_12_mo.L0 d1_occupancy_rate.L1 d1_cmbs_to_gdp.L0 d1_cmbs_to_gdp.L4 d1_fed_funds_rate.L4 d1_fed_funds_rate.L0

filtered_df = filtered_df[['d1_appreciation_return','d1_starts_sf_12_mo', 'd1_demand_sf', 'd1_net_absorp_sf_12_mo', 'd1_occupancy_rate', 'd1_cmbs_to_gdp', 'd1_fed_funds_rate']]


KeyError: "['d1_starts_sf_12_mo', 'd1_demand_sf', 'd1_net_absorp_sf_12_mo', 'd1_occupancy_rate'] not in index"

In [19]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ardl import ARDL
import warnings
warnings.filterwarnings("ignore")

# Assume 'filtered_df' is the final DataFrame from your previous steps.
# 'target_transformation' holds the name of the transformed target variable.
target = target_transformation
predictors = [var for var in filtered_df.columns if var != target]

# Set maximum lags to test for the AR component and for all predictors.
max_ar_lag = 4      # Maximum autoregressive lags for the target
max_exog_lag = 4    # Maximum lag order for each exogenous predictor

best_aic = np.inf
best_model = None
best_ar_lag = None
best_exog_lag = None

# Grid search over candidate lags:
for ar_lag in range(1, max_ar_lag + 1):
    for exog_lag in range(1, max_exog_lag + 1):
        try:
            # Build and fit the ARDL model using 'order' instead of 'exog_lags'
            model = ARDL(filtered_df[target], lags=ar_lag, exog=filtered_df[predictors], order=exog_lag)
            result = model.fit()
            
            # Evaluate the model using AIC
            if result.aic < best_aic:
                best_aic = result.aic
                best_model = result
                best_ar_lag = ar_lag
                best_exog_lag = exog_lag
        except Exception as e:
            print(f"Error with AR lag {ar_lag} and exogenous lag {exog_lag}: {e}")
            continue

print(f"Best ARDL model: AR lag = {best_ar_lag}, Exogenous lag = {best_exog_lag}, AIC = {best_aic:.4f}\n")
if best_model is not None:
    print(best_model.summary())
else:
    print("No ARDL model could be fitted. Check data and model specifications.")

# -------------------------------
# Forecasting with the ARDL Model
# -------------------------------
# Forecast horizons: 2 and 4 periods ahead.
# When forecasting, you need to supply values for the exogenous predictors.
# Here we assume the last observed exogenous values persist into the future.

# Retrieve the last row of exogenous data.
last_exog = filtered_df[predictors].iloc[-1:]
# For forecasting, replicate this row for the required forecast horizon.
forecast_horizon_2 = 2
exog_forecast_2 = pd.concat([last_exog] * forecast_horizon_2, ignore_index=True)

if best_model is not None:
    forecast_2 = best_model.forecast(steps=forecast_horizon_2, exog=exog_forecast_2)
    print(f"\nForecast for {forecast_horizon_2} periods ahead:")
    print(forecast_2)
else:
    print("No ARDL model available for forecasting.")

forecast_horizon_4 = 4
exog_forecast_4 = pd.concat([last_exog] * forecast_horizon_4, ignore_index=True)

if best_model is not None:
    forecast_4 = best_model.forecast(steps=forecast_horizon_4, exog=exog_forecast_4)
    print(f"\nForecast for {forecast_horizon_4} periods ahead:")
    print(forecast_4)


Best ARDL model: AR lag = 4, Exogenous lag = 3, AIC = -313.2583

                                  ARDL Model Results                                 
Dep. Variable:          orig_market_cap_rate   No. Observations:                   91
Model:             ARDL(4, 3, 3, 3, 3, 3, 3)   Log Likelihood                 186.629
Method:                      Conditional MLE   S.D. of innovations              0.028
Date:                       Fri, 20 Feb 2026   AIC                           -313.258
Time:                               19:38:18   BIC                           -239.281
Sample:                                    4   HQIC                          -283.470
                                          91                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                         0.1678      0.062      2.712 

In [ ]:
#print model results for only those coeffcieints that have a p-value less than 0.05, so that i can see 

In [20]:
# For the forecast 4 outcome, display only the coefficients from the ARDL model that have a p-value less than 0.05.
significant_params = best_model.params[best_model.pvalues < 0.05]
print("\nSignificant coefficients (p-value < 0.05):")
print(significant_params)


Significant coefficients (p-value < 0.05):
const                        0.167788
orig_market_cap_rate.L1      1.333680
orig_market_cap_rate.L4     -0.361177
d1_appreciation_return.L0   -0.041576
d1_appreciation_return.L2    0.058528
d1_appreciation_return.L3   -0.031066
d1_asking_rent_growth.L0    -0.025334
dtype: float64
